### Plotting CHMM outputs 

run in osld enviroment 

In [1]:
import os
os.chdir("/home/anna-beer/Documents/anna_phd/Canonical-HMM-Networks") #sets working directory to the repo, so that all imports work correctly
print(os.getcwd())
from glob import glob
import numpy as np
import pickle

from osl_dynamics.analysis import power
from osl_dynamics.utils import plotting
from osl_dynamics.analysis import connectivity
import matplotlib.pyplot as plt 

/home/anna-beer/Documents/anna_phd/Canonical-HMM-Networks


In [ ]:
#PLOTTING CHMM OUTPUTS

#open hmm directory
base = "/rdrives/DRS-foundation-brain/zoe_data/BIDS"
deriv = f"{base}/derivatives_anna_01042026"
hmm_dir = f"{deriv}/hmm_8state_finetuned"
alp_dir = f"{hmm_dir}/alp"
alp_dir_pkl = f"{hmm_dir}/alp_pkl"
stc_dir = f"{hmm_dir}/stc"
figs_dir =f"{hmm_dir}/figures"
summary_stats_dir =f"{hmm_dir}/summary_stats"


# load the data
alp = pickle.load(open(f"{alp_dir_pkl}/alp.pkl", "rb"))
f = np.load(f"{summary_stats_dir}/f.npy")
psd = np.load(f"{summary_stats_dir}/psd.npy")
coh = np.load(f"{summary_stats_dir}/coh.npy")
w = np.load(f"{summary_stats_dir}/w.npy")
pow_maps = np.load(f"{summary_stats_dir}/pow_maps.npy")
coh_nets = np.load(f"{summary_stats_dir}/coh_nets.npy")
fo = np.load(f"{summary_stats_dir}/fo.npy")
lt = np.load(f"{summary_stats_dir}/lt.npy")
intv = np.load(f"{summary_stats_dir}/intv.npy")
sr = np.load(f"{summary_stats_dir}/sr.npy")

# Average over subjects and channels
gpsd = np.average(psd, axis=0)#, weights=w)
psd_mean = np.mean(gpsd, axis=1)
print(psd_mean.shape)

# # Plot
n_states = psd_mean.shape[0]
plotting.plot_line(
    [f] * n_states,
    psd_mean,
    labels=[f"State {i}" for i in range(1, n_states + 1)],
    x_label="Frequency (Hz)",
    y_label="PSD (a.u.)",
    x_range=[f[0], f[-1]],
    filename = f"{figs_dir}/group_avg_psd.png"
)


##
p = power.variance_from_spectra(f, psd)
p_mean = np.average(p, axis=0)#, weights=w)


power.save(
    p_mean,
    mask_file="MNI152_T1_8mm_brain.nii.gz",
    parcellation_file="Glasser52_binary_space-MNI152NLin6_res-8x8x8.nii.gz",
    filename= f"{figs_dir}/power_maps.png"
)

# Takes a few seconds for the power maps to appear
power.save(
    p_mean,
    mask_file="MNI152_T1_8mm_brain.nii.gz",
    parcellation_file="Glasser52_binary_space-MNI152NLin6_res-8x8x8.nii.gz",
    subtract_mean=True,
    filename= f"{figs_dir}/powermaps_minusmean.png",
)


##
c = connectivity.mean_coherence_from_spectra(f, coh)
mean_c = np.average(c, axis=0)#, weights=w)
mean_c -= np.mean(mean_c, axis=0)
connectivity.save(
    mean_c,
    parcellation_file="Glasser52_binary_space-MNI152NLin6_res-8x8x8.nii.gz",
    filename=f"{figs_dir}/connectivity_networks.png", 
)

thres_mean_c = connectivity.threshold(mean_c, percentile=97, absolute_value=True)

connectivity.save(
    thres_mean_c,
    parcellation_file="Glasser52_binary_space-MNI152NLin6_res-8x8x8.nii.gz",
    filename=f"{figs_dir}/connectivity_networks_thresholded.png"
)

# Plot the distribution of fractional occupancy (FO) across subjects
plotting.plot_violin(fo.T, x_label="State", y_label="Fractional Occupancy", filename=f"{figs_dir}/fo_violin_plot.png")
plotting.plot_violin(fo.T, x_label="State", y_label="Fractional Occupancy")
plt.ylim(0, 0.4)  # set limits here
plt.savefig(f"{figs_dir}/fo_violin_plot-rescaled.png")
# # Plot distribution across subjects
plotting.plot_violin(lt.T, x_label="State", y_label="Mean Lifetime (s)", filename=f"{figs_dir}/lt_violin_plot.png")
plotting.plot_violin(lt.T, x_label="State", y_label="Mean Lifetime (s)")
plt.ylim(0, 0.4)  # set limits here
plt.savefig(f"{figs_dir}/lt_violin_plot-rescaled.png")
# # Plot distribution across subjects
plotting.plot_violin(intv.T, x_label="State", y_label="Mean Interval (s)", filename=f"{figs_dir}/intv_violin_plot.png")
plotting.plot_violin(intv.T, x_label="State", y_label="Mean Interval (s)")
plt.ylim(0, 5.25)  # set limits here
plt.savefig(f"{figs_dir}/intv_violin_plot-rescaled.png")
# # Plot distribution across subjects
plotting.plot_violin(sr.T, x_label="State", y_label="Switching Rate (Hz)", filename=f"{figs_dir}/sr_violin_plot.png")
plotting.plot_violin(sr.T, x_label="State", y_label="Switching Rate (Hz)")
plt.ylim(0, 5)  # set limits here
plt.savefig(f"{figs_dir}/sr_violin_plot-rescaled.png")

(8, 90)


In [3]:
psd_global_mean = np.mean(psd_mean, axis=0)  # shape: (n_freqs,)
n_states = psd_mean.shape[0]

import matplotlib.pyplot as plt

colors = ['#1f77b4', 'g', 'brown', 'r', 'purple', 'darkblue', 'darkgreen', 'orange']

for i in range(n_states):
    plt.figure()

    # State PSD
    plt.plot(f, psd_mean[i], label=f"State {i+1}", color=colors[i])

    # Global mean PSD (black dashed)
    plt.plot(f, psd_global_mean, 'k--', label="Mean PSD")

    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD (a.u.)")
    plt.xlim([f[0], f[-1]])
    plt.ylim(0,0.14)
    plt.legend()

    plt.savefig(f"{figs_dir}/psd_state_{i+1}.png")
    plt.close()
